### Structured output
>Models can be requested towprovide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")

model

/Users/chinmaybhatt/agenticai/langchain/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [18]:
from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title:str=Field(description="The Title of the Movie")
    year:int=Field(description="This Year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:str=Field(description="This Movie rating out of 10")

In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11098c590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11098d010>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The Title of the Movie', 'type': 'string'}, 'year': {'description': 'This Year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'This Movie rating out of 10', 'type': 'string'}}, 'required': ['title', 'year', 'director', 'ra

In [16]:
# for chunk in model.stream("Provide the details about Jujtsu Kaisen"):
#     print(chunk.text, end="", flush=True)

# model.invoke("Provide the details about Jujtsu Kaisen")
model_with_structure.invoke("Provide the details about Jujtsu Kaisen")

Movie(title='Jujutsu Kaisen: Infinite Train', year=2023, director='Takahashi', rating='9.0')

### Message output alongside Parsed structure

In [28]:
from pydantic import BaseModel, Field 

class Movie(BaseModel):
    """A Movie With Details"""
    title:str=Field(description="The Title of the Movie")
    year:int=Field(description="This Year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="This Movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

responses = model_with_structure.invoke("Provide the details about Inception")
responses


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a Movie function that requires title, year, director, and rating. I need to fill those parameters. I know Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and rating. Yes, 2010 is correct. The rating is 8.8. So I'll structure the tool call with those details.\n", 'tool_calls': [{'id': '63y9vj4nv', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 162, 'prompt_tokens': 229, 'total_tokens': 391, 'completion_time': 0.292041865, 'completion_tokens_details': {'reasoning_tokens': 114}, 'prompt_time': 0.008987271, 'prompt_tokens_details': None, 'queue_time': 0.053883823, 'total_ti

### Nested Structure

In [29]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str


class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in Million USD")
    

In [32]:
model_with_structure = model.with_structured_output(MovieDetails)

responses = model_with_structure.invoke("Provide details about the movie Jujutsu Kaisen")
responses

MovieDetails(title='Jujutsu Kaisen 0', year=2021, cast=[Actor(name='Megumi Toyogosato', role='Yuta Okkotsu'), Actor(name='Kaiji Tang', role='Mahito'), Actor(name='Atsumi Tanezaki', role='Nobara Kugisaki')], genres=['Action', 'Fantasy', 'Horror'], budget=None)

### TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [36]:
from typing_extensions import TypedDict, Annotated

In [54]:
class MovieDict(TypedDict):
    """A Movie with Details"""
    title: Annotated[str, ..., "The Title of The Movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The year the movie was released"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [55]:
model_with_typedict = model.with_structured_output(MovieDict)

reponses = model_with_typedict.invoke("Please provide the details of the movie avengers endgame")

reponses

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [57]:
class Actor(TypedDict):
    name: str
    role: str


class MovieDetails(TypedDict):
    title: str
    year: int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in Million USD")


model_with_typedict = model.with_structured_output(MovieDetails)

reponses = model_with_typedict.invoke("Please provide the details of the movie avengers endgame")

reponses
    

{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Christian Bale', 'role': 'Bruce Wayne / Batman'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers: Endgame',
 'year': 2019}

In [58]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### Data Classes
A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [60]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [66]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact Informations For a Person"""
    name: str = Field(description="The Name of the person")
    email: str = Field(description="The Email addresed of the Person")
    phone: str = Field(description="The Phone Number of the Person")


agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='5918eea8-d41e-432d-9de3-aa760ece9bf8'),
  AIMessage(content='{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d827c-7852-7c72-aab3-2c80cf9dab44-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [67]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [70]:
result["messages"]

[HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='5918eea8-d41e-432d-9de3-aa760ece9bf8'),
 AIMessage(content='{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d827c-7852-7c72-aab3-2c80cf9dab44-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})]